<a href="https://colab.research.google.com/github/omonigho-egbo-15/Data_Science--ITDS-/blob/main/GroupLab6_Titanic_Feature_Engineering_through_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Group Lab 6 - Titanic Feature Engineering through EDA (Project Group 1)
Alexander Debusk | Anashrah Adil | Omonigho Egbo | Thien Nam Nguyen

Run top-to-bottom. It:
  1. loads the data (CSV or the original .xlsx) and audits missing values,
  2. prints the EDA stats we used to justify each feature,
  3. saves one chart per engineered feature into ./charts/ (for the video),
  4. runs the model after EVERY phase and feature addition (the "climb"),
  5. prints a final summary table.

Model: LogisticRegression (standardized, L2) - binary target, small tabular data,
signed interpretable coefficients for the forensic narrative.
Metric: 5-fold stratified cross-validated accuracy (classes are 62/38, so
accuracy is meaningful; we watch F1 too).
"""

import os
import re
import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use("Agg")          # comment out when running live in Jupyter
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

RNG = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
CHART_DIR = "charts"
os.makedirs(CHART_DIR, exist_ok=True)

# ---------------------------------------------------------------- load data
def load_data():
    # First look for local files
    for path in ["titanic_train.csv", "titantic_train.csv.xlsx",
                 "titanic_train.csv.xlsx", "titanic.csv"]:
        if os.path.exists(path):
            if path.endswith(".csv"):
                return pd.read_csv(path), path
            return pd.read_excel(path, engine="openpyxl"), path

    # Fallback: Download from a reliable public repository if local file is missing
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    print(f"Local file not found. Downloading dataset from {url}...")
    df_downloaded = pd.read_csv(url)
    df_downloaded.to_csv("titanic_train.csv", index=False)
    return df_downloaded, "titanic_train.csv"

df, source = load_data()
print(f"Loaded {len(df)} rows x {df.shape[1]} cols from {source}\n")
print("Missing values per column:")
print(df.isna().sum().to_string(), "\n")
y = df["Survived"].values

# ---------------------------------------------------------------- phase 1
RAW_FEATURES = ["Pclass", "SibSp", "Parch", "Fare"]   # numeric, no missing
# PassengerId deliberately excluded: a row index, not a signal.

def evaluate(label, X):
    """5-fold stratified CV accuracy of standardized logistic regression."""
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("lr", LogisticRegression(max_iter=2000, random_state=RNG))])
    scores = cross_val_score(pipe, X, y, cv=CV, scoring="accuracy")
    print(f"{label:<55} acc = {scores.mean():.4f}  (folds: {np.round(scores, 4)}, std {scores.std():.4f})")
    return scores.mean()

results = []
def record(name, acc, prev):
    results.append({"model": name, "cv_accuracy": round(acc, 4),
                    "delta": round(acc - prev, 4) if prev is not None else None})
    return acc

print("=" * 80)
print("PHASE 1 - RAW BASELINE (numeric columns, no missing data)")
print("=" * 80)
acc_raw = record("Raw baseline (Pclass, SibSp, Parch, Fare)",
                 evaluate("Raw baseline", df[RAW_FEATURES]), None)

# ---------------------------------------------------------------- phase 2
print()
print("=" * 80)
print("PHASE 2 - ORDERED CLEANING (imputation & outlier handling)")
print("=" * 80)
clean = df.copy()

# Step 1: impute Embarked with the mode (tiny gap, fix first so all 891 rows
# are usable for every statistic computed afterwards).
clean["Embarked"] = clean["Embarked"].fillna(clean["Embarked"].mode()[0])

# Step 2: impute Age with the median of its Pclass x Sex group (177 rows).
# Group medians respect the strong class/age and sex/age structure.
clean["Age"] = clean.groupby(["Pclass", "Sex"])["Age"].transform("median")

# Step 3: cap Fare outliers at the 99th percentile (right-skewed: median ~14.45,
# max 512.33 -> winsorizing keeps the signal without letting 9 extreme tickets
# dominate the linear fit).
fare_p99 = clean["Fare"].quantile(0.99)
clean["Fare"] = clean["Fare"].clip(upper=fare_p99)
print(f"Fare 99th percentile used as cap: {fare_p99:.3f}")

# Step 4: drop identifier / noise columns (PassengerId, Ticket).
clean = clean.drop(columns=["PassengerId", "Ticket"])

# Step 5: encode categoricals (encoding is LAST: imputation and capping are
# safer on the original columns).
clean["Sex"] = clean["Sex"].map({"male": 0, "female": 1})
clean["Embarked_S"] = (clean["Embarked"] == "S").astype(int)
clean["Embarked_Q"] = (clean["Embarked"] == "Q").astype(int)

CLEAN_FEATURES = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",
                  "Embarked_S", "Embarked_Q"]
acc_clean = record("Clean baseline",
                   evaluate("Clean baseline (+Sex, Age imputed, Fare capped, Embarked)",
                            clean[CLEAN_FEATURES]), acc_raw)
print(f"\nCleaning change vs raw: {acc_clean - acc_raw:+.4f}")

# ---------------------------------------------------------------- EDA / phase 3
print()
print("=" * 80)
print("PHASE 3 - FEATURE ENGINEERING (one feature at a time)")
print("=" * 80)

def rate_by(col, data=None):
    d = data if data is not None else clean
    return d.groupby(col)["Survived"].agg(["count", "mean"])

def bar_chart(col, title, fname, order=None, data=None):
    stats = rate_by(col, data)
    if order:
        stats = stats.reindex(order)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    bars = ax.bar(stats.index.astype(str), stats["mean"] * 100, color="#4C72B0")
    for b, (n, m) in zip(bars, stats[["count", "mean"]].values):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1.5,
                f"{m*100:.1f}%\n(n={n})", ha="center", fontsize=9)
    ax.set_title(title)
    ax.set_ylabel("Survival rate (%)")
    ax.set_ylim(0, 100)
    fig.tight_layout()
    fig.savefig(os.path.join(CHART_DIR, fname), dpi=150)
    plt.close(fig)
    print(f"saved charts/{fname}")

# ---- overview chart (context for the video intro)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ["Sex", "Pclass"]):
    s = rate_by(col)
    ax.bar(s.index.astype(str), s["mean"] * 100, color="#55A868")
    ax.set_title(f"Survival by {col}")
    ax.set_ylabel("%")
    for i, (n, m) in enumerate(s[["count", "mean"]].values):
        ax.text(i, m * 100 + 1.5, f"{m*100:.1f}% (n={n})", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(CHART_DIR, "fig1_overview.png"), dpi=150)
plt.close(fig)
print("saved charts/fig1_overview.png")

# ---- feature 1: Title ------------------------------------------------------
# Observation: the manifest embeds a social title in every name. Titles split
# the population far more sharply than raw Sex/Age - especially "Master",
# which labels BOYS (mean age ~4.6, max 12) inside the near-fatal male group.
clean["Title"] = clean["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
clean["TitleGroup"] = clean["Title"].where(
    clean["Title"].isin(["Mr", "Miss", "Mrs", "Master"]), "Rare")
print("\nSurvival and age by title:")
print(pd.concat([rate_by("TitleGroup"),
                 clean.groupby("TitleGroup")["Age"].mean().rename("mean_age")], axis=1))
bar_chart("TitleGroup", "Survival rate by title (Master = boys)",
          "fig2_title.png", order=["Mr", "Miss", "Mrs", "Master", "Rare"])
for t in ["Mr", "Miss", "Mrs", "Master"]:
    clean[f"Title_{t}"] = (clean["TitleGroup"] == t).astype(int)

acc1 = record("Clean + Title",
             evaluate("Step 1: + Title one-hot",
                      clean[CLEAN_FEATURES + ["Title_Mr", "Title_Miss",
                                             "Title_Mrs", "Title_Master"]]), acc_clean)

# ---- feature 2: FamilySize --------------------------------------------------
# Observation: survival by family size is an inverted U - 1: 30.4%, 2: 55.3%,
# 3: 57.8%, 4: 72.4%, 5+: ~15%. Small families help; giant families do not.
clean["FamilySize"] = clean["SibSp"] + clean["Parch"] + 1
print("\nSurvival by family size:")
print(rate_by("FamilySize"))
bar_chart("FamilySize", "Survival rate by family size (inverted U)",
          "fig3_family_size.png")
S2 = CLEAN_FEATURES + ["Title_Mr", "Title_Miss", "Title_Mrs", "Title_Master",
                       "FamilySize"]
acc2 = record("Clean + Title + FamilySize", evaluate("Step 2: + FamilySize", clean[S2]), acc1)

# ---- feature 3: IsAlone -----------------------------------------------------
# Observation: solo travelers survived at 30.4% vs 50.6% with family.
clean["IsAlone"] = (clean["FamilySize"] == 1).astype(int)
print("\nSurvival alone vs with family:")
print(rate_by("IsAlone").rename(index={0: "with family", 1: "alone"}))
bar_chart("IsAlone", "Survival rate: alone vs with family", "fig4_is_alone.png")
S3 = S2 + ["IsAlone"]
acc3 = record("Clean + ... + IsAlone", evaluate("Step 3: + IsAlone", clean[S3]), acc2)

# ---- feature 4: Has_Cabin ---------------------------------------------------
# Observation: a recorded cabin is a wealth/deck proxy: 66.7% survival when the
# cabin is known vs 30.0% when missing (687 missing). We keep the reliable part
# of a mostly-missing column: was a cabin recorded at all?
clean["Has_Cabin"] = clean["Cabin"].notna().astype(int)
print("\nSurvival by cabin-known:")
print(rate_by("Has_Cabin").rename(index={0: "unknown cabin", 1: "known cabin"}))
bar_chart("Has_Cabin", "Survival rate by cabin recorded", "fig5_cabin.png")
S4 = S3 + ["Has_Cabin"]
acc4 = record("Clean + ... + Has_Cabin", evaluate("Step 4: + Has_Cabin", clean[S4]), acc3)

# ---- feature 5: Child -------------------------------------------------------
# Observation: children <= 12 survived at 58.0% vs 36.7% - "women and children
# first" in the data. We flag children explicitly.
clean["Child"] = (clean["Age"] <= 12).astype(int)
print("\nSurvival child (<=12) vs others:")
print(rate_by("Child").rename(index={0: "adult", 1: "child <= 12"}))
bar_chart("Child", "Survival rate: children (<=12) vs adults", "fig6_child.png")
S5 = S4 + ["Child"]
acc5 = record("Clean + ... + Child", evaluate("Step 5: + Child", clean[S5]), acc4)

# ---------------------------------------------------------------- summary
print()
print("=" * 80)
print("THE FEATURE CLIMB - SUMMARY")
print("=" * 80)
print(pd.DataFrame(results).to_string(index=False))
print(f"""
Key numbers:
  Raw -> Clean:            {acc_clean - acc_raw:+.4f}  (cleaning)
  Clean -> best engineered: {max(acc1, acc2, acc3, acc4, acc5) - acc_clean:+.4f}
  Biggest single engineered gain: Title {acc1 - acc_clean:+.4f}
""")
print("Note: scores are 5-fold CV means with seed 42; small (+/-0.01) shifts")
print("across library versions are normal. Re-running this script refreshes")
print("every number in the report - keep the report and video consistent.")
